# A08 · Attention, Line by Line

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a08_attention.ipynb)

Attention lets every token look at every earlier token and take a weighted average of their information. You'll compute it by hand on a 3-token example, prove why it's scaled by √d_k, build multi-head causal self-attention in PyTorch, and count its quadratic cost.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a08). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Compute causal self-attention by hand for 3 tokens.
- Derive why scores are divided by $\sqrt{d_k}$.
- Implement multi-head causal self-attention and match `F.scaled_dot_product_attention`.
- Count attention FLOPs and memory, $O(T^2 d)$, and explain what FlashAttention changes.

## The math

### Scaled dot-product attention

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}} + M\right)V, \qquad M_{ij} = \begin{cases}0 & j \le i\\ -\infty & j > i\end{cases}$$

Three tokens, $d_k = 2$: $Q = K = \begin{pmatrix}1&0\\0&1\\1&1\end{pmatrix}$, $V = \begin{pmatrix}1&0\\0&2\\1&1\end{pmatrix}$. Look at token 3.

**Worked example**

1. $q_3 K^{\top} = (1,\ 1,\ 2)$, divided by $\sqrt2$: $(0.707,\ 0.707,\ 1.414)$ (token 3 can see all three (mask allows j ≤ 3))
2. softmax $\Rightarrow (0.248,\ 0.248,\ 0.503)$ (weights sum to 1)
3. output $= 0.248(1,0) + 0.248(0,2) + 0.503(1,1) = (0.752,\ 1.000)$
4. token 2 sees only tokens 1–2: weights $(0.330,\ 0.670)$, output $(0.330,\ 1.340)$ (token 1 attends only to itself)

### Why divide by √d_k

$$q, k \sim \mathcal{N}(0, I_{d_k}) \;\Rightarrow\; \mathrm{Var}(q^{\top}k) = \sum_{i=1}^{d_k}\mathrm{Var}(q_ik_i) = d_k$$

Unscaled dot products have standard deviation $\sqrt{d_k}$. With $d_k = 128$, scores of ±20 push softmax into near one-hot outputs with vanishing gradients.

**Worked example**

1. $d_k = 64$: std of $q^\top k$ is 8; after dividing by $\sqrt{64} = 8$ it's 1
2. softmax(0, 8) = (0.0003, 0.9997) vs softmax(0, 1) = (0.27, 0.73) (saturated vs trainable)

### Multi-head and cost

$$\mathrm{MHA}(X) = \mathrm{Concat}(\text{head}_1,\dots,\text{head}_h)\,W_O,\quad \text{FLOPs}_{\text{scores+AV}} \approx 4\,T^2 d$$

Projections cost $8Td^2$ FLOPs (four $d\times d$ matmuls); the $QK^\top$ and $AV$ matmuls cost $2T^2d$ each.

**Worked example**

1. GPT-2 small: $d = 768,\ h = 12 \Rightarrow d_{head} = 64$
2. $T = 1024$: projections $8Td^2 = 4.8$ GFLOPs, scores+AV $4T^2d = 3.2$ GFLOPs per layer (comparable at this length)
3. $T = 8192$, fp16: one head's score matrix $= 8192^2\cdot2$ B $= 134$ MB (per head, per layer: FlashAttention avoids storing it)

## Lab: Attention by hand, then multi-head in PyTorch

Match the worked example exactly, then build the attention layer you'll reuse in GPT.

### Step 1

The worked example, then PyTorch's fused kernel.

In [ ]:
import torch, math, torch.nn as nn, torch.nn.functional as F
Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
K = Q.clone()
V = torch.tensor([[1., 0.], [0., 2.], [1., 1.]])
scores = Q @ K.T / math.sqrt(Q.shape[-1])
mask = torch.tril(torch.ones(3, 3, dtype=torch.bool))
weights = scores.masked_fill(~mask, float('-inf')).softmax(dim=-1)
print(weights)        # row 3: [0.2483, 0.2483, 0.5035]
print(weights @ V)    # row 3: [0.7517, 1.0000]
print(F.scaled_dot_product_attention(Q[None], K[None], V[None], is_causal=True)[0])

### Step 2

Why $\sqrt{d_k}$: measure dot-product variance with and without scaling.

In [ ]:
for d in [4, 64, 1024]:
    q, k = torch.randn(10000, d), torch.randn(10000, d)
    dots = (q * k).sum(-1)
    print(f"d={d:5d}  var raw={dots.var():8.1f}  var scaled={(dots / math.sqrt(d)).var():.2f}")

### Step 3

Multi-head causal self-attention, the module used in GPT (A09).

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head, self.d_head = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, d_head)
        q = q.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)        # (B, h, T, T)
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
        att = att.masked_fill(~mask, float('-inf')).softmax(dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)        # re-assemble heads
        return self.proj(y)

attn = CausalSelfAttention(d_model=64, n_head=4)
x = torch.randn(2, 10, 64)
print(attn(x).shape)                                   # (2, 10, 64)
# causality test: changing a future token must not change earlier outputs
x2 = x.clone(); x2[:, -1] += 1.0
print(torch.allclose(attn(x)[:, :-1], attn(x2)[:, :-1], atol=1e-6))   # True

### Step 4

Swap in the fused kernel (FlashAttention on supported GPUs) and check it agrees.

In [ ]:
def fast_forward(self, x):
    B, T, C = x.shape
    q, k, v = self.qkv(x).split(C, dim=2)
    q, k, v = (t.view(B, T, self.n_head, self.d_head).transpose(1, 2) for t in (q, k, v))
    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))
print(torch.allclose(attn(x), fast_forward(attn, x), atol=1e-5))     # True

### Clean up

Nothing to clean up.

## Check yourself

**1. Scores for one query are $(2, 0, -\infty)$. Attention weights?**

A. (0.5, 0.5, 0)  
B. (0.881, 0.119, 0)  
C. (1, 0, 0)  
D. (0.67, 0.33, 0)

<details><summary>Answer</summary>

**B.** $e^2/(e^2 + e^0) = 7.389/8.389 = 0.881$; the masked entry gets $e^{-\infty} = 0$.

</details>

**2. $d_{model} = 4096$ with 32 heads. Head dimension?**

A. 32  
B. 64  
C. 128  
D. 4096

<details><summary>Answer</summary>

**C.** $4096/32 = 128$.

</details>

**3. Doubling the context length from 4k to 8k multiplies attention-score compute by…**

A. 2  
B. 4  
C. 8  
D. 1

<details><summary>Answer</summary>

**B.** Score computation is $O(T^2)$: $(8/4)^2 = 4$.

</details>